# Dropout and Normalization: Two Layers Every Diagram Has

The full code of [Dropout and Normalization: Two Layers Every Diagram Has](https://residual-stream.lvtan71.workers.dev/posts/dropout-and-normalization/), run from [its folder](https://github.com/lvtan71/residual-stream-code/tree/main/posts/dropout-and-normalization) in the
order the folder's README gives. Each script is followed by the output it
printed for the post, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/dropout-and-normalization.ipynb)

**Hardware:** Runs on a CPU; Colab's free runtime is enough. About 96 min end to end on the RTX 3090 machine that saved the outputs below.


The post's numbers come from an Apple M4 laptop. Other hardware and library
versions change timings and the last digits of some floating-point results.
Sampled text and small training runs can move further; where that changes a
number the post discusses, the note above says so.

**Outputs saved from a run on NVIDIA GeForce RTX 3090 on 2026-10-18** (torch 2.14.1+cu130, 95 min; every cell ran). Run the notebook to replace them with your own.

In [1]:
%pip install -q "numpy==2.*"

# Setup: fetch the companion repo and work in this post's folder.
import os, subprocess, sys
SLUG = "dropout-and-normalization"
ROOT = os.path.abspath("..") if os.path.isdir("../posts") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
os.chdir(os.path.join(ROOT, "posts", SLUG))
print("working in", os.getcwd())

Note: you may need to restart the kernel to use updated packages.
working in residual-stream-code/posts/dropout-and-normalization


## `grad_check.py`

The whole network's backward pass, with no norm, LayerNorm and RMSNorm, against finite differences.

In [2]:
!{sys.executable} grad_check.py

norm=None: largest relative error 5.9e-08


norm=layer: largest relative error 5.0e-08


norm=rms: largest relative error 4.9e-07


The post printed:

```text
norm=None: largest relative error 3.8e-08
norm=layer: largest relative error 6.7e-08
norm=rms: largest relative error 4.9e-07
```

## `dropout_grid.py`

A two-hidden-layer router trained with Adam for 3,000 steps (or the step count given as its argument) at 5 dropout rates × 5 AdamW weight decays on 20 splits: validation loss, accuracy and confidence, the setting chosen on validation, and its test loss.

In [3]:
!{sys.executable} dropout_grid.py

3,000 steps of Adam. Differences are paired, against no dropout at the same decay.
dropout 0.0  decay 0.0  val CE 1.115                  val acc 0.667                  mean top p 0.857  top-2 logit gap  3.85  top p when wrong 0.752
dropout 0.1  decay 0.0  val CE 1.228  +0.113 ± 0.052  val acc 0.680  +0.013 ± 0.023  mean top p 0.874  top-2 logit gap  4.43  top p when wrong 0.786
dropout 0.3  decay 0.0  val CE 1.462  +0.347 ± 0.080  val acc 0.686  +0.019 ± 0.032  mean top p 0.910  top-2 logit gap  5.80  top p when wrong 0.840
dropout 0.5  decay 0.0  val CE 1.661  +0.546 ± 0.132  val acc 0.679  +0.011 ± 0.027  mean top p 0.926  top-2 logit gap  6.93  top p when wrong 0.860
dropout 0.7  decay 0.0  val CE 1.513  +0.397 ± 0.116  val acc 0.684  +0.016 ± 0.029  mean top p 0.913  top-2 logit gap  6.54  top p when wrong 0.843
dropout 0.0  decay 0.3  val CE 0.953                  val acc 0.683                  mean top p 0.807  top-2 logit gap  2.95  top p when wrong 0.711
dropout 0.1  decay 0.3 

The post printed:

```text
3,000 steps of Adam. Differences are paired, against no dropout at the same decay.
dropout 0.0  decay 0.0  val CE 1.115                  val acc 0.667                  mean top p 0.857  top-2 logit gap  3.85  top p when wrong 0.752
dropout 0.1  decay 0.0  val CE 1.228  +0.113 ± 0.052  val acc 0.680  +0.013 ± 0.023  mean top p 0.874  top-2 logit gap  4.43  top p when wrong 0.786
dropout 0.3  decay 0.0  val CE 1.462  +0.347 ± 0.080  val acc 0.686  +0.019 ± 0.032  mean top p 0.910  top-2 logit gap  5.80  top p when wrong 0.840
dropout 0.5  decay 0.0  val CE 1.661  +0.546 ± 0.132  val acc 0.679  +0.011 ± 0.027  mean top p 0.926  top-2 logit gap  6.93  top p when wrong 0.860
dropout 0.7  decay 0.0  val CE 1.513  +0.397 ± 0.116  val acc 0.684  +0.016 ± 0.029  mean top p 0.913  top-2 logit gap  6.54  top p when wrong 0.843
dropout 0.0  decay 0.3  val CE 0.953                  val acc 0.683                  mean top p 0.807  top-2 logit gap  2.95  top p when wrong 0.711
dropout 0.1  decay 0.3  val CE 0.950  -0.003 ± 0.034  val acc 0.693  +0.010 ± 0.027  mean top p 0.822  top-2 logit gap  3.18  top p when wrong 0.719
dropout 0.3  decay 0.3  val CE 0.974  +0.021 ± 0.044  val acc 0.704  +0.021 ± 0.038  mean top p 0.845  top-2 logit gap  3.58  top p when wrong 0.751
dropout 0.5  decay 0.3  val CE 1.018  +0.065 ± 0.058  val acc 0.704  +0.021 ± 0.040  mean top p 0.864  top-2 logit gap  3.99  top p when wrong 0.772
dropout 0.7  decay 0.3  val CE 1.016  +0.062 ± 0.060  val acc 0.701  +0.019 ± 0.034  mean top p 0.864  top-2 logit gap  4.12  top p when wrong 0.767
dropout 0.0  decay 1.0  val CE 0.879                  val acc 0.680                  mean top p 0.762  top-2 logit gap  2.41  top p when wrong 0.658
dropout 0.1  decay 1.0  val CE 0.869  -0.011 ± 0.046  val acc 0.689  +0.009 ± 0.024  mean top p 0.787  top-2 logit gap  2.67  top p when wrong 0.675
dropout 0.3  decay 1.0  val CE 0.865  -0.014 ± 0.054  val acc 0.698  +0.017 ± 0.041  mean top p 0.792  top-2 logit gap  2.76  top p when wrong 0.683
dropout 0.5  decay 1.0  val CE 0.846  -0.033 ± 0.057  val acc 0.715  +0.035 ± 0.034  mean top p 0.793  top-2 logit gap  2.76  top p when wrong 0.692
dropout 0.7  decay 1.0  val CE 0.809  -0.070 ± 0.056  val acc 0.714  +0.034 ± 0.032  mean top p 0.775  top-2 logit gap  2.55  top p when wrong 0.662
dropout 0.0  decay 3.0  val CE 0.809                  val acc 0.694                  mean top p 0.719  top-2 logit gap  1.94  top p when wrong 0.605
dropout 0.1  decay 3.0  val CE 0.806  -0.003 ± 0.017  val acc 0.690  -0.004 ± 0.020  mean top p 0.715  top-2 logit gap  1.93  top p when wrong 0.596
dropout 0.3  decay 3.0  val CE 0.799  -0.010 ± 0.022  val acc 0.689  -0.005 ± 0.024  mean top p 0.705  top-2 logit gap  1.86  top p when wrong 0.582
dropout 0.5  decay 3.0  val CE 0.781  -0.028 ± 0.025  val acc 0.707  +0.014 ± 0.025  mean top p 0.678  top-2 logit gap  1.68  top p when wrong 0.561
dropout 0.7  decay 3.0  val CE 0.806  -0.004 ± 0.035  val acc 0.698  +0.004 ± 0.025  mean top p 0.606  top-2 logit gap  1.25  top p when wrong 0.492
dropout 0.0  decay 10.0 val CE 0.940                  val acc 0.670                  mean top p 0.482  top-2 logit gap  0.67  top p when wrong 0.411
dropout 0.1  decay 10.0 val CE 0.962  +0.022 ± 0.018  val acc 0.647  -0.022 ± 0.028  mean top p 0.469  top-2 logit gap  0.62  top p when wrong 0.403
dropout 0.3  decay 10.0 val CE 1.021  +0.081 ± 0.027  val acc 0.629  -0.041 ± 0.043  mean top p 0.430  top-2 logit gap  0.46  top p when wrong 0.376
dropout 0.5  decay 10.0 val CE 1.104  +0.164 ± 0.031  val acc 0.620  -0.050 ± 0.042  mean top p 0.380  top-2 logit gap  0.33  top p when wrong 0.342
dropout 0.7  decay 10.0 val CE 1.246  +0.306 ± 0.036  val acc 0.443  -0.227 ± 0.042  mean top p 0.312  top-2 logit gap  0.07  top p when wrong 0.304
chosen on validation: dropout 0.5, decay 3.0; without dropout: decay 3.0
test CE: 0.776 chosen, 0.798 decay only, 1.083 neither. Dropout's own share: -0.022 ± 0.024; decay's: -0.285 ± 0.097
```

## `dropout_grid_long.py`

Runs `dropout_grid.py 12000`: the same grid, four times as long.

In [4]:
!{sys.executable} dropout_grid_long.py

12,000 steps of Adam. Differences are paired, against no dropout at the same decay.
dropout 0.0  decay 0.0  val CE 1.467                  val acc 0.689                  mean top p 0.903  top-2 logit gap  5.55  top p when wrong 0.833
dropout 0.1  decay 0.0  val CE 1.787  +0.320 ± 0.085  val acc 0.686  -0.003 ± 0.025  mean top p 0.924  top-2 logit gap  7.21  top p when wrong 0.862
dropout 0.3  decay 0.0  val CE 2.516  +1.049 ± 0.173  val acc 0.691  +0.003 ± 0.024  mean top p 0.954  top-2 logit gap 10.68  top p when wrong 0.914
dropout 0.5  decay 0.0  val CE 2.847  +1.380 ± 0.324  val acc 0.706  +0.018 ± 0.022  mean top p 0.960  top-2 logit gap 13.47  top p when wrong 0.925
dropout 0.7  decay 0.0  val CE 3.533  +2.066 ± 0.481  val acc 0.696  +0.008 ± 0.030  mean top p 0.965  top-2 logit gap 16.41  top p when wrong 0.933
dropout 0.0  decay 0.3  val CE 1.294                  val acc 0.654                  mean top p 0.850  top-2 logit gap  4.14  top p when wrong 0.775
dropout 0.1  decay 0.3

The post printed:

```text
12,000 steps of Adam. Differences are paired, against no dropout at the same decay.
dropout 0.0  decay 0.0  val CE 1.467                  val acc 0.689                  mean top p 0.903  top-2 logit gap  5.55  top p when wrong 0.833
dropout 0.1  decay 0.0  val CE 1.787  +0.320 ± 0.085  val acc 0.686  -0.003 ± 0.025  mean top p 0.924  top-2 logit gap  7.21  top p when wrong 0.862
dropout 0.3  decay 0.0  val CE 2.516  +1.049 ± 0.173  val acc 0.691  +0.003 ± 0.024  mean top p 0.954  top-2 logit gap 10.68  top p when wrong 0.914
dropout 0.5  decay 0.0  val CE 2.847  +1.380 ± 0.324  val acc 0.706  +0.018 ± 0.022  mean top p 0.960  top-2 logit gap 13.47  top p when wrong 0.925
dropout 0.7  decay 0.0  val CE 3.533  +2.066 ± 0.481  val acc 0.696  +0.008 ± 0.030  mean top p 0.965  top-2 logit gap 16.41  top p when wrong 0.933
dropout 0.0  decay 0.3  val CE 1.294                  val acc 0.654                  mean top p 0.850  top-2 logit gap  4.14  top p when wrong 0.775
dropout 0.1  decay 0.3  val CE 1.230  -0.064 ± 0.098  val acc 0.696  +0.043 ± 0.052  mean top p 0.883  top-2 logit gap  4.84  top p when wrong 0.802
dropout 0.3  decay 0.3  val CE 1.326  +0.031 ± 0.122  val acc 0.688  +0.034 ± 0.048  mean top p 0.892  top-2 logit gap  5.42  top p when wrong 0.805
dropout 0.5  decay 0.3  val CE 1.296  +0.001 ± 0.116  val acc 0.695  +0.041 ± 0.044  mean top p 0.902  top-2 logit gap  5.71  top p when wrong 0.816
dropout 0.7  decay 0.3  val CE 1.241  -0.053 ± 0.131  val acc 0.714  +0.060 ± 0.035  mean top p 0.900  top-2 logit gap  5.71  top p when wrong 0.824
dropout 0.0  decay 1.0  val CE 1.086                  val acc 0.680                  mean top p 0.848  top-2 logit gap  3.74  top p when wrong 0.760
dropout 0.1  decay 1.0  val CE 1.111  +0.025 ± 0.079  val acc 0.691  +0.011 ± 0.028  mean top p 0.854  top-2 logit gap  3.99  top p when wrong 0.767
dropout 0.3  decay 1.0  val CE 1.029  -0.057 ± 0.048  val acc 0.695  +0.015 ± 0.017  mean top p 0.844  top-2 logit gap  3.83  top p when wrong 0.738
dropout 0.5  decay 1.0  val CE 0.919  -0.167 ± 0.058  val acc 0.699  +0.019 ± 0.019  mean top p 0.834  top-2 logit gap  3.48  top p when wrong 0.716
dropout 0.7  decay 1.0  val CE 0.842  -0.244 ± 0.065  val acc 0.713  +0.033 ± 0.022  mean top p 0.808  top-2 logit gap  2.99  top p when wrong 0.697
dropout 0.0  decay 3.0  val CE 1.010                  val acc 0.667                  mean top p 0.806  top-2 logit gap  2.90  top p when wrong 0.710
dropout 0.1  decay 3.0  val CE 0.922  -0.087 ± 0.037  val acc 0.688  +0.020 ± 0.014  mean top p 0.792  top-2 logit gap  2.73  top p when wrong 0.683
dropout 0.3  decay 3.0  val CE 0.814  -0.195 ± 0.045  val acc 0.700  +0.033 ± 0.023  mean top p 0.751  top-2 logit gap  2.28  top p when wrong 0.631
dropout 0.5  decay 3.0  val CE 0.774  -0.236 ± 0.064  val acc 0.706  +0.039 ± 0.028  mean top p 0.703  top-2 logit gap  1.85  top p when wrong 0.579
dropout 0.7  decay 3.0  val CE 0.799  -0.210 ± 0.083  val acc 0.708  +0.040 ± 0.029  mean top p 0.609  top-2 logit gap  1.27  top p when wrong 0.495
dropout 0.0  decay 10.0 val CE 0.958                  val acc 0.663                  mean top p 0.470  top-2 logit gap  0.62  top p when wrong 0.405
dropout 0.1  decay 10.0 val CE 0.957  -0.001 ± 0.021  val acc 0.662  +0.000 ± 0.031  mean top p 0.468  top-2 logit gap  0.58  top p when wrong 0.402
dropout 0.3  decay 10.0 val CE 1.027  +0.069 ± 0.024  val acc 0.650  -0.013 ± 0.038  mean top p 0.425  top-2 logit gap  0.43  top p when wrong 0.375
dropout 0.5  decay 10.0 val CE 1.121  +0.164 ± 0.033  val acc 0.636  -0.026 ± 0.041  mean top p 0.368  top-2 logit gap  0.29  top p when wrong 0.332
dropout 0.7  decay 10.0 val CE 1.255  +0.298 ± 0.039  val acc 0.426  -0.236 ± 0.056  mean top p 0.306  top-2 logit gap  0.05  top p when wrong 0.298
chosen on validation: dropout 0.5, decay 3.0; without dropout: decay 10.0
test CE: 0.782 chosen, 0.973 decay only, 1.368 neither. Dropout's own share: -0.191 ± 0.041; decay's: -0.395 ± 0.167
```

## `batchnorm_flaws.py`

BatchNorm's four problems on one layer's pre-activations: dependence on the batch, noisy small-batch statistics, a batch of one, and leaking across a sequence.

In [5]:
!{sys.executable} batchnorm_flaws.py

one question, two batches: its BatchNorm outputs differ by 0.37 on average (their typical size is 0.91)
batch of   2: batch mean is off by 0.55 standard deviations on average
batch of   8: batch mean is off by 0.28 standard deviations on average
batch of  32: batch mean is off by 0.13 standard deviations on average


batch of 128: batch mean is off by 0.04 standard deviations on average
batch of 1: BatchNorm output 0.0 everywhere; LayerNorm output still varies, size 0.77
change the last token only: first token's output moves by 1.294 with BatchNorm, 0.000 with LayerNorm


The post printed:

```text
one question, two batches: its BatchNorm outputs differ by 0.37 on average (their typical size is 0.91)
batch of   2: batch mean is off by 0.55 standard deviations on average
batch of   8: batch mean is off by 0.28 standard deviations on average
batch of  32: batch mean is off by 0.13 standard deviations on average
batch of 128: batch mean is off by 0.04 standard deviations on average
batch of 1: BatchNorm output 0.0 everywhere; LayerNorm output still varies, size 0.77
change the last token only: first token's output moves by 1.294 with BatchNorm, 0.000 with LayerNorm
```

## `norms.py`

LayerNorm, RMSNorm and BatchNorm with their backward passes (helper); run, it prints the worked vectors and checks both backward passes against finite differences.

In [6]:
!{sys.executable} norms.py

x = [2. 4. 6. 8.]:  LayerNorm [-1.342 -0.447  0.447  1.342]   RMSNorm [0.365 0.73  1.095 1.461]
x = [102. 104. 106. 108.]:  LayerNorm [-1.342 -0.447  0.447  1.342]   RMSNorm [0.971 0.99  1.009 1.028]


LayerNorm backward vs finite differences: largest relative error 1.9e-09
RMSNorm backward vs finite differences: largest relative error 2.4e-10


The post printed:

```text
x = [2. 4. 6. 8.]:  LayerNorm [-1.342 -0.447  0.447  1.342]   RMSNorm [0.365 0.73  1.095 1.461]
x = [102. 104. 106. 108.]:  LayerNorm [-1.342 -0.447  0.447  1.342]   RMSNorm [0.971 0.99  1.009 1.028]
LayerNorm backward vs finite differences: largest relative error 1.9e-09
RMSNorm backward vs finite differences: largest relative error 2.4e-10
```

## `norm_cost.py`

The time for one LayerNorm or RMSNorm on an H100, math against memory, next to one matmul.

In [7]:
!{sys.executable} norm_cost.py

LayerNorm math    32 us   memory: unfused 4 passes 0.64 ms, fused 2 passes 0.32 ms
RMSNorm   math    20 us   memory: unfused 3 passes 0.48 ms, fused 2 passes 0.32 ms
one 4096 x 4096 matmul on the same tokens, at tensor-core peak: 2.22 ms


The post printed:

```text
LayerNorm math    32 us   memory: unfused 4 passes 0.64 ms, fused 2 passes 0.32 ms
RMSNorm   math    20 us   memory: unfused 3 passes 0.48 ms, fused 2 passes 0.32 ms
one 4096 x 4096 matmul on the same tokens, at tensor-core peak: 2.22 ms
```

## `deep_signal.py`

The 20-layer test of "Initialization and Optimizers" with a LayerNorm before every ReLU, at three starting scales.

In [8]:
!{sys.executable} deep_signal.py

weight std   activation size, layer 20   gradient reaching layer 1   gradient on W, layer 1


     0.010                        0.71                    7.84e-01                 1.11e+02


     0.088                        0.71                    7.85e-01                 1.25e+01


     1.000                        0.71                    7.85e-01                 1.11e+00


The post printed:

```text
weight std   activation size, layer 20   gradient reaching layer 1   gradient on W, layer 1
     0.010                        0.71                    7.84e-01                 1.11e+02
     0.088                        0.71                    7.85e-01                 1.25e+01
     1.000                        0.71                    7.85e-01                 1.11e+00
```

## `norm_lr_grid.py`

The ten-layer router from std 0.01 and from He, with no norm, LayerNorm and RMSNorm, swept over learning rates for SGD and Adam.

In [9]:
!{sys.executable} norm_lr_grid.py

start     norm   optimizer  train loss after 1,000 steps, by learning rate


std 0.01  none   SGD    0.0001: 1.386  0.001: 1.386  0.01: 1.386  0.1: 1.387  1: 1.392
std 0.01  none   Adam   1e-05: 1.386  0.0001: 1.386  0.001: 1.386  0.01: 1.387
std 0.01  layer  SGD    0.0001: 0.872  0.001: 0.069  0.01: 1.387  0.1: 1.387  1: 1.392
std 0.01  layer  Adam   1e-05: 0.808  0.0001: 0.072  0.001: 1.386  0.01: 1.387
std 0.01  rms    SGD    0.0001: 0.798  0.001: 0.062  0.01: 1.386  0.1: 1.387  1: 1.392
std 0.01  rms    Adam   1e-05: 0.722  0.0001: 0.055  0.001: 1.143  0.01: 1.387
He        none   SGD    0.0001: 1.385  0.001: 1.286  0.01: 0.002  0.1: 0.000  1: inf
He        none   Adam   1e-05: 1.230  0.0001: 0.001  0.001: 0.000  0.01: 0.000
He        layer  SGD    0.0001: 1.214  0.001: 0.226  0.01: 0.006  0.1: 0.001  1: 1.392
He        layer  Adam   1e-05: 0.803  0.0001: 0.010  0.001: 0.000  0.01: 0.221
He        rms    SGD    0.0001: 1.221  0.001: 0.207  0.01: 0.005  0.1: 0.001  1: 1.392
He        rms    Adam   1e-05: 0.810  0.0001: 0.009  0.001: 0.000  0.01: 0.000


The post printed:

```text
start     norm   optimizer  train loss after 1,000 steps, by learning rate
std 0.01  none   SGD    0.0001: 1.386  0.001: 1.386  0.01: 1.386  0.1: 1.387  1: 1.392
std 0.01  none   Adam   1e-05: 1.386  0.0001: 1.386  0.001: 1.386  0.01: 1.387
std 0.01  layer  SGD    0.0001: 0.872  0.001: 0.069  0.01: 1.387  0.1: 1.387  1: 1.392
std 0.01  layer  Adam   1e-05: 0.808  0.0001: 0.072  0.001: 1.386  0.01: 1.387
std 0.01  rms    SGD    0.0001: 0.798  0.001: 0.062  0.01: 1.386  0.1: 1.387  1: 1.392
std 0.01  rms    Adam   1e-05: 0.722  0.0001: 0.055  0.001: 1.196  0.01: 1.387
He        none   SGD    0.0001: 1.385  0.001: 1.286  0.01: 0.002  0.1: 0.000  1: inf
He        none   Adam   1e-05: 1.230  0.0001: 0.001  0.001: 0.000  0.01: 0.000
He        layer  SGD    0.0001: 1.214  0.001: 0.226  0.01: 0.006  0.1: 0.001  1: 1.392
He        layer  Adam   1e-05: 0.803  0.0001: 0.010  0.001: 0.000  0.01: 0.506
He        rms    SGD    0.0001: 1.221  0.001: 0.207  0.01: 0.005  0.1: 0.001  1: 1.392
He        rms    Adam   1e-05: 0.810  0.0001: 0.009  0.001: 0.000  0.01: 0.000
```